# 🏥 IMO Coding Intelligence API — Notebook 3: CMS Excludes 1 Conflict Check

This notebook demonstrates the **CMS Excludes 1** detection capability of the IMO Coding Intelligence API.

### What is Excludes 1?
> An **Excludes 1** note in ICD-10-CM means the two conditions **cannot be coded together** on the same claim — they are considered mutually exclusive. Submitting conflicting code pairs can result in claim denials or audits.

**Example:** `C18.1` (Malignant neoplasm of appendix) has an Excludes 1 note for `C7A.024` (Malignant carcinoid tumor of the appendix). They cannot be coded simultaneously.

---

### What you'll cover
1. Authenticate and configure the Excludes 1 API endpoint
2. Define a set of ICD-10-CM codes that include known conflict pairs
3. POST codes to the `cms-excludes1` endpoint
4. Parse the `results[].note_codes[]` response structure
5. Display a clear conflict report

> 💡 **Note:** The Excludes 1 API only applies to `ICD-10-CM` codes. Other code systems are ignored by this endpoint.

## Step 1: Setup — Imports & Authentication

In [ ]:
import requests
import json
import os
import pandas as pd

# ── Credentials ───────────────────────────────────────────────
CLIENT_ID     = os.environ.get("IMO_CODING_INTEL_CLIENT_ID",     "YOUR_CLIENT_ID_HERE")
CLIENT_SECRET = os.environ.get("IMO_CODING_INTEL_CLIENT_SECRET", "YOUR_CLIENT_SECRET_HERE")

AUTH_URL      = "https://api.imohealth.com/oauth/token"
EXCLUDES1_URL = "https://api.imohealth.com/codingintelligence/v1/rules/cms-excludes1"

def get_access_token(client_id, client_secret):
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "audience":      "https://api.imohealth.com"
    }
    r = requests.post(AUTH_URL, json=payload, timeout=30)
    r.raise_for_status()
    return r.json()["access_token"]

ACCESS_TOKEN = get_access_token(CLIENT_ID, CLIENT_SECRET)
HEADERS = {
    "Content-Type":  "application/json",
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}
print("✓ Authenticated")

## Step 2: Define ICD-10-CM Codes to Check

The pairs below are known to trigger Excludes 1 conflicts. You can swap these out for your own codes. The API evaluates **all codes as a group** — it checks every code against every other code in the submitted list.

In [ ]:
# Known Excludes 1 conflict pairs — the API will detect these
icd_codes = [
    {"code": "C18.1",   "code_system": "ICD-10-CM"},   # Malignant neoplasm of appendix
    {"code": "C7A.024", "code_system": "ICD-10-CM"},   # Malignant carcinoid tumor of appendix ← conflicts with C18.1
    {"code": "M12.542", "code_system": "ICD-10-CM"},   # Intermittent hydrarthrosis, left hand
    {"code": "M17.2",   "code_system": "ICD-10-CM"},   # Bilateral primary osteoarthritis of knee ← conflicts with M12.542
    {"code": "I10",     "code_system": "ICD-10-CM"},   # Essential hypertension — no conflict expected
]

print(f"✓ {len(icd_codes)} ICD-10-CM codes loaded")
for c in icd_codes:
    print(f"   {c['code']}")

## Step 3: Understanding the API Payload

The `cms-excludes1` endpoint requires a specific payload format. Each code entry must include a `record_id` field (a unique string identifier for the code within this submission). The API uses this to trace back which submitted code is in conflict.

In [ ]:
# Build the payload — each code gets an auto-assigned record_id
payload = {
    "codes": [
        {
            "code":        c["code"],
            "code_system": c["code_system"],
            "record_id":   str(i + 1)          # required by the API spec
        }
        for i, c in enumerate(icd_codes)
    ]
}

print("Payload preview:")
print(json.dumps(payload, indent=2))

## Step 4: Call the cms-excludes1 Endpoint

In [ ]:
response = requests.post(EXCLUDES1_URL, headers=HEADERS, json=payload, timeout=30)
response.raise_for_status()

data          = response.json()
results       = data.get("results", [])
total_results = data.get("total_results", 0)

print(f"✓ API call successful")
print(f"  Total results returned : {total_results}")
print(f"\nRaw response:")
print(json.dumps(data, indent=2))

## Step 5: Parse & Display Conflicts

The response returns a `results` array. Each entry in `results` represents a code that has at least one Excludes 1 conflict. The `note_codes` field lists the other codes in the submission that conflict with it.

In [ ]:
print("=" * 65)
print("CMS EXCLUDES 1 CONFLICT REPORT")
print("=" * 65)

if not results:
    print("\n✅ No Excludes 1 conflicts detected — all codes are compatible.")
else:
    for r in results:
        code        = r.get("code", "—")
        note_codes  = r.get("note_codes", [])
        conflicting = [nc.get("code") for nc in note_codes if nc.get("code")]

        print(f"\n  ⛔ {code}")
        print(f"     Conflicts with : {', '.join(conflicting)}")
        print(f"     These codes cannot be billed together on the same claim.")

print(f"\n{'=' * 65}")
print(f"  {len(results)} conflict(s) found across {len(icd_codes)} submitted codes")
print("=" * 65)

In [ ]:
# ── Pandas conflict table ────────────────────────────────────
conflict_rows = []

# Map submitted codes for quick lookup
submitted_map = {c["code"]: c["code_system"] for c in icd_codes}

for r in results:
    code       = r.get("code", "—")
    note_codes = r.get("note_codes", [])
    for nc in note_codes:
        conflict_rows.append({
            "Code":             code,
            "Code System":      submitted_map.get(code, "ICD-10-CM"),
            "Conflicts With":   nc.get("code", "—"),
            "Conflict System":  nc.get("code_system", "ICD-10-CM"),
            "CMS Rule":         "Excludes 1 — Cannot code together"
        })

if conflict_rows:
    df_conflicts = pd.DataFrame(conflict_rows)
    display(df_conflicts)
else:
    print("✅ No conflicts — nothing to display")

## ✅ Summary

| Step | Description |
|------|-------------|
| 1 | Authenticated with IMO OAuth (Coding Intelligence credentials) |
| 2 | Defined ICD-10-CM codes including known Excludes 1 conflict pairs |
| 3 | Built payload with required `record_id` per code |
| 4 | POSTed to `cms-excludes1` endpoint |
| 5 | Parsed `results[].note_codes[]` to identify conflicting pairs |
| 6 | Displayed conflict report as console output and pandas DataFrame |

---

### What to do with conflicts
When an Excludes 1 conflict is detected, a coder should:
1. **Review the clinical documentation** — confirm whether both conditions are truly present
2. **Remove one of the conflicting codes** if they are mutually exclusive
3. **Consult ICD-10-CM guidelines** for the specific Excludes 1 note if clinical context justifies both conditions

---

**You've completed all three notebooks!** 🎉

| Notebook | Topic |
|----------|-------|
| 01 | Setup & Authentication |
| 02 | Upload Codes & Validate Billing |
| 03 | CMS Excludes 1 Conflict Detection |